# r3con — provenance

**TL;DR** — r3con records where an answer came from:

- the answer cites the relevance snippets and the parsed records it used, separately
  (`result.provenance["summary_ids"]` and `result.provenance["parse_ids"]`);
- each relevance snippet cites the other documents' snippets it used (`result.relevance_provenance`);
- each document's parse cites the snippets it used (`result.parsing_provenance`);
- each parsed record names its document (`"document"` in `result.structured_context`);
- the answer quotes the sentences behind it (`result.provenance["quotes"]`): those of the
  snippets it cites, of the snippets its parses cite, and of the earlier snippets they cite.

They cite documents by id, a part of a document read in parts as `<id>.<k>`, and
`result.document_ids[i]` is the id of `documents[i]`.
The memos and the question are the [quickstart](quickstart.ipynb)'s.

In [1]:
import os
from pathlib import Path

from r3con import r3con

# The key comes from the provider's own variable, or a .env beside this notebook.
assert os.environ.get("OPENAI_API_KEY") or os.path.exists(".env"), (
    "Set OPENAI_API_KEY first."
)

MODEL = "openai/gpt-6-luna"  # any litellm model string

QUESTION = (
    "Which maintenance contractor's sites logged the most equipment incidents in Q3 "
    "in total, and how many? Give the contractor's name, not its code."
)

In [2]:
docs = r3con.read_documents("memos")
result = r3con.run(question=QUESTION, documents=docs, model=MODEL)
print(result)

Halloran Services Ltd’s sites logged the most equipment incidents in Q3, with 11 in total.


In [3]:
# read_documents reads a folder in sorted path order, so names[i] is docs[i]'s file.
names = [p.name for p in sorted(Path("memos").glob("*.txt"))]
name_of = dict(zip(result.document_ids, names, strict=True))

# a part of a document read in parts is cited as <id>.<k>
summaries = [name_of[id_.split(".")[0]] for id_ in result.provenance["summary_ids"]]
parses = [name_of[id_] for id_ in result.provenance["parse_ids"]]

print("ids                 :", name_of)
print("answer's summaries  :", summaries)
print("answer's parses     :", parses)

ids                 : {'96c563': '01_northgate.txt', '01e955': '02_eastfield.txt', '42ce83': '03_riverside.txt', '1aca4c': '04_contractor_registry.txt', 'cd1d3a': '05_cafeteria.txt'}
answer's summaries  : ['01_northgate.txt', '02_eastfield.txt', '03_riverside.txt', '04_contractor_registry.txt']
answer's parses     : ['01_northgate.txt', '02_eastfield.txt', '03_riverside.txt', '04_contractor_registry.txt']


In [4]:
summarized = [ref.split(".")[0] for ref in result.provenance["summary_ids"]]
for id_ in dict.fromkeys(summarized):
    i = result.document_ids.index(id_)
    summary = result.relevant_context[i].strip().partition("\n")[0]
    drew_on = [name_of[ref.split(".")[0]] for ref in result.relevance_provenance[i]]
    print(f"{name_of[id_]}\n  summary : {summary[:100]}\n  drew on : {drew_on}\n")

01_northgate.txt
  summary : A Northgate Q3 operations memo reports 5 equipment incidents and identifies maintenance contractor c
  drew on : ['04_contractor_registry.txt']

02_eastfield.txt
  summary : A Q3 operations memo for the Eastfield site. It reports 9 equipment incidents and says maintenance i
  drew on : ['04_contractor_registry.txt']

03_riverside.txt
  summary : A Riverside site Q3 operations memo reports 6 equipment incidents and identifies contractor code CT-
  drew on : ['01_northgate.txt', '04_contractor_registry.txt']

04_contractor_registry.txt
  summary : An approved contractor registry extract that maps site-paperwork codes to contractor names: CT-118 i
  drew on : ['01_northgate.txt', '02_eastfield.txt', '03_riverside.txt']



In [5]:
for field, records in result.structured_context.items():
    for record in records:
        if record.get("document") in result.provenance["parse_ids"]:
            print(f"{name_of[record['document']]}  {field}: {record}")

01_northgate.txt  site_incident_reports: {'document': '96c563', 'site': 'Northgate', 'period': 'Q3', 'equipment_incident_count': 5, 'maintenance_contractor_code': 'CT-118', 'maintenance_contractor_name': None}
02_eastfield.txt  site_incident_reports: {'document': '01e955', 'site': 'Eastfield', 'period': 'Q3', 'equipment_incident_count': 9, 'maintenance_contractor_code': 'CT-204', 'maintenance_contractor_name': None}
03_riverside.txt  site_incident_reports: {'document': '42ce83', 'site': 'Riverside', 'period': 'Q3', 'equipment_incident_count': 6, 'maintenance_contractor_code': 'CT-118', 'maintenance_contractor_name': None}
04_contractor_registry.txt  contractor_code_mappings: {'document': '1aca4c', 'contractor_code': 'CT-118', 'contractor_name': 'Halloran Services Ltd'}
04_contractor_registry.txt  contractor_code_mappings: {'document': '1aca4c', 'contractor_code': 'CT-204', 'contractor_name': 'Merrow Facilities Group'}
04_contractor_registry.txt  contractor_code_mappings: {'document': '

In [6]:
# start and end place each quote in its document: docs[i][start:end] is the sentence.
quotes = result.provenance["quotes"]
for quote in quotes:
    print(f"{name_of[quote['document']]:<27} {quote['text']}")

first = quotes[0]
i = result.document_ids.index(first["document"])
print("\ndocs[i][start:end]          :", docs[i][first["start"] : first["end"]])

01_northgate.txt            Equipment incidents logged this quarter: 5.
01_northgate.txt            Planned maintenance at this site is carried out under contractor code CT-118.
02_eastfield.txt            Equipment incidents logged this quarter: 9.
02_eastfield.txt            Planned maintenance at this site is carried out under contractor code CT-204.
03_riverside.txt            Equipment incidents logged this quarter: 6.
03_riverside.txt            Planned maintenance at this site is carried out under contractor code CT-118.
04_contractor_registry.txt  CT-118 .... Halloran Services Ltd — mechanical and electrical maintenance
04_contractor_registry.txt  CT-204 .... Merrow Facilities Group — HVAC and compressed air
04_contractor_registry.txt  CT-311 .... Bellweather Grounds — landscaping (seasonal)
04_contractor_registry.txt  Codes are used in site paperwork; the registry is the authority for the mapping.

docs[i][start:end]          : Equipment incidents logged this quarter: 5.


In the run folder, `result.run_dir`, the same ids are in `ids.json` (id → index in `documents`), `relevance/result.json` (each round's relevance snippets, the quotes of their document they are based on, and what they cite), `structuring/parsing/result.json` (the records and what each document's parse cites) and `reasoning/result.json` (the answer, the snippets and parses it cites, and the quotes behind it).